The Shared Responsibility Model defines the boundary between what AWS secures and what customers are responsible for. Compliance covers the programmes, tools, and services that help you meet regulatory requirements on AWS. This topic also covers the key security posture services — GuardDuty, Inspector, Macie, and Security Hub — which detect threats and vulnerabilities across your AWS environment. These concepts appear throughout the SAA-C03 exam, often as the deciding factor between answer choices.

## The Shared Responsibility Model

AWS and the customer share responsibility for security, but the boundary depends on the service type.

```text
┌─────────────────────────────────────────────────┐
│           CUSTOMER RESPONSIBILITY               │
│     "Security IN the cloud"                     │
│                                                 │
│  Data (encryption, classification, integrity)   │
│  IAM (users, roles, policies, MFA)              │
│  Applications and platform                      │
│  OS patching (EC2)                              │
│  Network config (Security Groups, NACLs)        │
│  Client-side encryption                         │
├─────────────────────────────────────────────────┤
│           AWS RESPONSIBILITY                    │
│     "Security OF the cloud"                     │
│                                                 │
│  Physical data centres (access, power, cooling) │
│  Hardware (servers, storage, networking)        │
│  Hypervisor and virtualisation layer            │
│  Managed service infrastructure                 │
│  Global network infrastructure                  │
└─────────────────────────────────────────────────┘
```

### How responsibility shifts by service type

| Responsibility | EC2 (IaaS) | RDS (Managed) | S3 (Object store) | Lambda (Serverless) |
|---|---|---|---|---|
| Physical security | AWS | AWS | AWS | AWS |
| Hypervisor | AWS | AWS | AWS | AWS |
| OS patching | **Customer** | AWS | AWS | AWS |
| Database patching | **Customer** | AWS | N/A | N/A |
| Runtime | **Customer** | AWS | N/A | AWS |
| Network firewall (SG) | **Customer** | **Customer** | N/A | N/A |
| Data encryption | **Customer** | **Customer** | **Customer** | **Customer** |
| IAM | **Customer** | **Customer** | **Customer** | **Customer** |

> **Key exam pattern:** As you move from IaaS → PaaS → SaaS, AWS takes on more responsibility. The customer is **always** responsible for their data, IAM configuration, and how they use the service.

## Shared Responsibility Examples

### EC2 — maximum customer responsibility
- AWS: physical host, hypervisor, network infrastructure
- Customer: OS installation and patching, application code, security groups, IAM roles attached to instance, data encryption, firewall rules

### RDS — AWS handles the database engine
- AWS: OS patching, database engine patching, hardware, backups (if enabled), Multi-AZ replication
- Customer: database user management, schema design, security group rules, encryption at rest (enable at creation), IAM authentication, network placement

### S3 — AWS manages all infrastructure
- AWS: hardware, storage durability, global infrastructure
- Customer: bucket policies, ACLs, public access block settings, versioning, encryption settings, data classification, replication config

### Lambda — AWS manages the runtime
- AWS: OS, runtime environment, scaling, hardware
- Customer: function code, IAM execution role, VPC config, environment variables (secrets management), resource-based policies

### The customer is always responsible for
1. **Data** — content stored in AWS, classification, encryption
2. **Identity** — IAM users, roles, policies, MFA enforcement
3. **Application security** — code vulnerabilities, dependency management
4. **How they configure services** — a misconfigured S3 public bucket is the customer's responsibility

## AWS Compliance Programmes

AWS maintains compliance with a wide range of industry and regulatory standards. AWS's **certifications and attestations** cover the infrastructure — your applications may additionally need their own certifications.

### Key compliance programmes

| Programme | Scope |
|---|---|
| **SOC 1 / 2 / 3** | Service Organisation Controls — financial reporting (SOC 1), security/availability/confidentiality (SOC 2), public summary (SOC 3) |
| **PCI DSS** | Payment Card Industry — required for processing credit card data |
| **HIPAA** | Healthcare data privacy (US) — AWS is HIPAA-eligible; you sign a BAA |
| **ISO 27001** | International information security management standard |
| **ISO 27017 / 27018** | Cloud security / cloud privacy |
| **FedRAMP** | US federal government cloud authorisation |
| **GDPR** | EU data protection regulation — AWS provides DPA and tools |
| **FIPS 140-2** | US cryptographic standard — KMS HSMs are FIPS 140-2 validated |
| **CSA STAR** | Cloud Security Alliance certification |

> AWS is **not** automatically HIPAA-compliant for your workload — you must configure your services correctly, sign a Business Associate Agreement (BAA) with AWS, and use only HIPAA-eligible services.

## AWS Artifact

AWS Artifact is a **self-service portal** for accessing AWS compliance reports and agreements — no need to contact AWS support.

### Two components

**Artifact Reports** — download AWS's own compliance documents:
- SOC 1, 2, 3 reports
- PCI DSS Attestation of Compliance
- ISO 27001 certificate
- FedRAMP authorisation packages
- GDPR-related documentation
- AWS penetration testing results

**Artifact Agreements** — review and accept legal agreements with AWS:
- **Business Associate Agreement (BAA)** — required for HIPAA workloads
- **GDPR Data Processing Addendum (DPA)**
- Can be applied to individual accounts or across all accounts in an Organisation

> **Exam tip:** "Audit report", "compliance certificate", "download SOC 2 report", "sign BAA for HIPAA" → AWS Artifact.

## Amazon GuardDuty

GuardDuty is a managed **threat detection** service that continuously monitors your AWS account for malicious activity and unauthorised behaviour using machine learning and threat intelligence.

### What GuardDuty analyses
- **CloudTrail logs** — detects unusual API calls, unauthorised deployments, root account usage
- **VPC Flow Logs** — detects unusual network traffic, port scanning, data exfiltration
- **DNS logs** — detects communication with known malicious domains (C2 callbacks)
- **EKS audit logs** — detects suspicious Kubernetes activity
- **S3 data events** — detects unusual S3 access patterns
- **RDS login events** — detects brute-force and unusual DB access
- **Lambda network activity** — detects unusual Lambda network calls
- **Malware protection** — scans EBS volumes and S3 objects for malware

### Key properties
- **Agentless** — no software to install; analyses log data that AWS already collects
- **30-day free trial** — no upfront commitment
- **Multi-account**: designate a GuardDuty administrator account to manage findings across an entire Organisation
- Findings delivered to GuardDuty console, EventBridge, and Security Hub
- Integrate with EventBridge → Lambda or SNS for automated remediation

### Example findings
- `UnauthorizedAccess:EC2/SSHBruteForce` — SSH brute force on EC2
- `Recon:EC2/PortProbeUnprotectedPort` — port scanning from an unusual IP
- `CryptoCurrency:EC2/BitcoinTool.B` — EC2 instance mining cryptocurrency
- `Exfiltration:S3/AnomalousBehavior` — unusual S3 data access

> **Exam tip:** "Detect threats", "identify malicious activity", "compromised EC2", "unusual API calls" → GuardDuty.

## Amazon Inspector

Inspector is an automated **vulnerability assessment** service — it scans your workloads for software vulnerabilities and unintended network exposure.

### What Inspector scans
- **EC2 instances** — OS and application package CVEs (via SSM Agent); network reachability (open ports, security group exposure)
- **ECR container images** — scans images when pushed to ECR; continuously re-scans when new CVEs are published
- **Lambda functions** — scans function code and layers for known vulnerabilities in dependencies

### Key properties
- **Agentless for network scanning** — uses VPC Flow Logs and security group config
- **SSM Agent required** for deep OS package scanning on EC2
- **Continuous scanning** — re-evaluates when new CVEs are discovered (not just on demand)
- Findings include a **risk score** and actionable remediation guidance
- Integrates with Security Hub and EventBridge
- Multi-account: centralise findings from all accounts in an Organisation

> **Exam tip:** "Vulnerability scanning", "CVE detection", "patch prioritisation", "container image scanning" → Inspector. Inspector finds vulnerabilities; GuardDuty detects active threats.

## Amazon Macie

Macie is a managed **data security and privacy** service that uses machine learning to discover and protect sensitive data stored in **S3**.

### What Macie does
- Automatically discovers **sensitive data** in S3: PII (names, addresses, SSNs, passport numbers), financial data (credit card numbers), credentials (API keys, passwords), healthcare data (PHI)
- Evaluates S3 bucket **security posture**: public access, unencrypted buckets, cross-account sharing, misconfigured ACLs
- Generates findings with severity levels and data samples
- Findings delivered to Security Hub and EventBridge

### Key properties
- **S3-only** — Macie specifically analyses S3; not EC2, RDS, or other stores
- Supports custom data identifiers — define your own regex patterns for organisation-specific sensitive data
- Multi-account: centralise Macie findings across an Organisation

> **Exam tip:** "Discover PII in S3", "sensitive data detection", "find unencrypted buckets with personal data" → Macie.

## AWS Security Hub

Security Hub aggregates, normalises, and prioritises security findings from multiple AWS security services and third-party tools into a **single pane of glass**.

### What Security Hub aggregates
- GuardDuty findings
- Inspector findings
- Macie findings
- IAM Access Analyzer findings
- AWS Firewall Manager findings
- AWS Config compliance results
- Third-party tools (CrowdStrike, Palo Alto, Splunk, etc.)

### Security standards and benchmarks
Security Hub continuously checks your environment against security best practices:
- **AWS Foundational Security Best Practices**
- **CIS AWS Foundations Benchmark**
- **PCI DSS**
- **NIST SP 800-53**

### Key properties
- Uses **AWS Security Finding Format (ASFF)** — normalised JSON schema for all findings
- **Automated response**: route findings to EventBridge → Lambda for auto-remediation
- Multi-account: aggregate findings from all Organisation accounts into a central account
- 30-day free trial

> **Exam tip:** "Single view of security findings", "aggregate security alerts across accounts", "check against security benchmarks" → Security Hub.

### Relationships between security services
```text
GuardDuty   ──┐
Inspector   ──┼──▶ Security Hub ──▶ EventBridge ──▶ Lambda (auto-remediation)
Macie       ──┘         │                           SNS (alerts)
Config      ──┘         ▼
                  Compliance dashboard
                  (findings + benchmarks)
```

## AWS Audit Manager

Audit Manager continuously collects evidence to help you **prepare for audits** — automatically mapping AWS resource configurations and usage to compliance framework controls.

### What it does
- Maps AWS API activity, resource snapshots, and Security Hub findings to controls in compliance frameworks (PCI DSS, SOC 2, HIPAA, ISO 27001, GDPR)
- Generates audit-ready evidence reports — no manual evidence collection
- Custom frameworks: define your own controls for internal audits

> **Exam tip:** "Prepare for audit", "collect compliance evidence automatically", "map AWS activity to PCI controls" → Audit Manager. Downloading AWS's own compliance reports → Artifact.

## Working with Compliance and Security Services via boto3

In [ ]:
import boto3
import json

guardduty  = boto3.client('guardduty',   region_name='us-east-1')
inspector  = boto3.client('inspector2',  region_name='us-east-1')
macie      = boto3.client('macie2',      region_name='us-east-1')
securityhub = boto3.client('securityhub', region_name='us-east-1')
artifact   = boto3.client('artifact',    region_name='us-east-1')

In [ ]:
# Enable GuardDuty
detector = guardduty.create_detector(
    Enable=True,
    FindingPublishingFrequency='FIFTEEN_MINUTES',  # SIX_HOURS | ONE_HOUR | FIFTEEN_MINUTES
    DataSources={
        'S3Logs':           {'Enable': True},
        'Kubernetes':       {'AuditLogs': {'Enable': True}},
        'MalwareProtection': {'ScanEc2InstanceWithFindings': {'EbsVolumes': True}}
    }
)
detector_id = detector['DetectorId']
print(f"GuardDuty enabled. Detector ID: {detector_id}")

# List current findings
findings = guardduty.list_findings(
    DetectorId=detector_id,
    FindingCriteria={
        'Criterion': {
            'severity': {'Gte': 7}   # HIGH (7+) and CRITICAL (9+) only
        }
    }
)
print(f"High/Critical findings: {len(findings['FindingIds'])}")

In [ ]:
# Enable Inspector v2 for EC2, ECR, and Lambda
inspector.enable(
    resourceTypes=['EC2', 'ECR', 'LAMBDA']
)
print("Inspector enabled for EC2, ECR, and Lambda")

# List critical vulnerabilities
findings = inspector.list_findings(
    filterCriteria={
        'severity': [{'comparison': 'EQUALS', 'value': 'CRITICAL'}]
    },
    maxResults=10
)
for finding in findings.get('findings', []):
    print(f"- {finding['title']} | {finding['awsAccountId']} | "
          f"Score: {finding.get('inspectorScore', 'N/A')}")

In [ ]:
# Enable Macie
macie.enable_macie(
    findingPublishingFrequency='FIFTEEN_MINUTES',
    status='ENABLED'
)
print("Macie enabled")

# Create a sensitive data discovery job for an S3 bucket
job = macie.create_classification_job(
    jobType='ONE_TIME',
    name='pii-scan-prod-bucket',
    s3JobDefinition={
        'bucketDefinitions': [{
            'accountId': '123456789012',
            'buckets': ['my-prod-data-bucket']
        }]
    },
    managedDataIdentifierSelector='ALL'   # use all AWS managed data identifiers
)
print(f"Macie scan job created: {job['jobId']}")

In [ ]:
# Enable Security Hub
securityhub.enable_security_hub(
    EnableDefaultStandards=True   # enables AWS Foundational Security Best Practices
)
print("Security Hub enabled")

# Enable additional standards
securityhub.batch_enable_standards(
    StandardsSubscriptionRequests=[
        {'StandardsArn': 'arn:aws:securityhub:us-east-1::standards/cis-aws-foundations-benchmark/v/1.4.0'},
        {'StandardsArn': 'arn:aws:securityhub:us-east-1::standards/pci-dss/v/3.2.1'}
    ]
)
print("CIS Benchmark and PCI DSS standards enabled")

# Get security score summary
standards = securityhub.get_enabled_standards()
for std in standards['StandardsSubscriptions']:
    print(f"Standard: {std['StandardsArn'].split('/')[-3]} | Status: {std['StandardsStatus']}")

In [ ]:
# List available compliance reports in AWS Artifact
reports = artifact.list_reports(maxResults=5)
for report in reports.get('reports', []):
    print(f"- {report['name']} | Category: {report['category']} | "
          f"Series: {report['series']}")

# Download a specific report (returns a presigned S3 URL)
# report_url = artifact.get_report_metadata(reportId='report-id-here')
# The URL is valid for a limited time and requires agreement acceptance

## Summary

| Concept | Key Takeaway |
|---|---|
| Shared Responsibility | AWS: security OF the cloud (hardware, infra, managed services); Customer: security IN the cloud (data, IAM, config) |
| IaaS (EC2) | Maximum customer responsibility — OS, runtime, patching, networking |
| PaaS (RDS) | AWS patches OS and DB engine; customer manages data, SGs, IAM, encryption |
| SaaS/Serverless (Lambda, S3) | AWS manages runtime/infra; customer always owns data and IAM |
| Always customer-owned | Data content, IAM configuration, how services are configured |
| AWS Artifact | Download AWS compliance reports (SOC, PCI, ISO); sign BAA for HIPAA, DPA for GDPR |
| HIPAA on AWS | Use HIPAA-eligible services + sign BAA via Artifact + configure correctly |
| GuardDuty | Threat detection; analyses CloudTrail, VPC Flow Logs, DNS; ML-based; agentless |
| GuardDuty findings | Active threats: brute force, crypto mining, C2 callbacks, unusual API calls |
| Inspector | Vulnerability scanning; CVEs in EC2/ECR/Lambda; continuous re-scan on new CVEs |
| Inspector vs GuardDuty | Inspector: known vulnerabilities (CVEs); GuardDuty: active threats and anomalies |
| Macie | S3-only; discovers PII and sensitive data; flags misconfigured buckets |
| Security Hub | Aggregates findings from GuardDuty, Inspector, Macie, Config; benchmark checks |
| Security Hub standards | AWS FSBP, CIS Benchmark, PCI DSS, NIST SP 800-53 |
| Audit Manager | Continuously collects evidence for audits; maps AWS activity to compliance controls |
| Artifact vs Audit Manager | Artifact: download AWS's compliance docs; Audit Manager: collect your own compliance evidence |